In [2]:
%reload_ext autoreload
%autoreload 2

from src.generate_surface_code import SurfaceCode
from src.TN_decoder import decoder
import stim
from matplotlib import pyplot as plt
import numpy as np
from tqdm import tqdm
from src.parse_syndrome import *
from src.coset_error_chains import *
from src.tensor_network_utils import contract_network
import pymatching

In [15]:
distance = 3
noise_model = "depolarize"
noise = 0.1
chi = 6
nshots = 1000

In [16]:
def count_logical_errors(code, num_shots: int) -> int:

    circuit = code.circuit
    sampler = circuit.compile_detector_sampler()
    detection_events, observable_flips = sampler.sample(shots = num_shots, separate_observables=True)
    predictions = []

    for event in tqdm(detection_events):
        predictions.append(decoder(code, event, chi))

    num_error = 0
    for shot in range(num_shots):
        actual_for_shot = observable_flips[shot][0]
        #print(actual_for_shot)
        predicted_for_shot = predictions[shot]
        if not actual_for_shot == predicted_for_shot:
            num_error += 1

    return num_error

In [17]:
def count_MWPM_logical_errors(circuit: stim.Circuit, num_shots: int) -> int:
    # Sample the circuit.
    sampler = circuit.compile_detector_sampler()
    detection_events, observable_flips = sampler.sample(num_shots, separate_observables=True)

    # Configure a decoder using the circuit.
    detector_error_model = circuit.detector_error_model(decompose_errors=True)
    matcher = pymatching.Matching.from_detector_error_model(detector_error_model)

    # Run the decoder.
    predictions = matcher.decode_batch(detection_events)

    # Count the mistakes.
    num_errors = 0
    for shot in range(num_shots):
        actual_for_shot = observable_flips[shot]
        predicted_for_shot = predictions[shot]
        if not np.array_equal(actual_for_shot, predicted_for_shot):
            num_errors += 1
    return num_errors

In [18]:
code = SurfaceCode(distance, noise_model, noise)

In [19]:
m_errors = count_MWPM_logical_errors(code.circuit, nshots)
print(f"There were {m_errors} wrong predictions out of {nshots} shots")

There were 79 wrong predictions out of 1000 shots


In [20]:
num_errors = count_logical_errors(code, nshots)
print(f"There were {num_errors} wrong predictions out of {nshots} shots")

100%|██████████| 1000/1000 [00:07<00:00, 135.47it/s]

There were 67 wrong predictions out of 1000 shots
